# B2B Revamp — Build `Analytics_Rpt.md.B2B_revamp_cohorts`

**Purpose:** For every `LoanNum × AsofDt` in `Analytics_Rpt.md.B2BPricingHist`, enrich with:

1. **Display attributes (as of `AsofDt`)** — current loan snapshot from `ADM.ODS.vwServLoanHist` plus the blended market rate.
2. **Baseline attributes (as of `CohortBaseDt`)** — loan attributes at the channel-specific pricing-decision date, held **static** for the life of the loan.
3. **Static cohort assignment** — `Rule_Bucket`, `Mean_CPR`, `Std_CPR`, `Cohort_ID`, `cohort_unique_ID` derived from the baseline attributes, so a loan keeps the same cohort across every snapshot month.

Cohort assignment uses the **schema version 4** artifacts documented in `cohort_README.md`
(`cohort_bands_config.json`, `prinbal_map.csv`, `spread_map.csv`, `cohort_definitions.csv`).
Cohort inputs are `LoanType`, `RemainingTerm`, `FICOOrig`, `PrinBal`, `IntRate`, `Market_Rate` (all `_Base`).
`InvestorCatgy` and `LoanAge` are **not** cohort inputs, but are still carried in the table.

> This notebook rebuilds `B2B_revamp_cohorts` (DROP + full rebuild). The older `B2B_revamp_cohorts_upd` table is not touched.

---

## Cohort anchor date (`CohortBaseDt`)

| `Channel` | Anchor date | `CohortBaseDtType` |
|---|---|---|
| `Correspondent` | `ADM.ODS.vwOrigLoan.LockDt` (falls back to `FundDt` if null) | `LockDt` |
| `Co-Issue` | `FundDt` | `FundDt` |
| `Consumer Direct` | `FundDt` | `FundDt` |

Baseline servicing attributes are read at `EOMONTH(CohortBaseDt)`; if that month-end snapshot does not
exist yet (common for Correspondent, where the lock precedes boarding) the query cascades forward up to
3 months to the first available snapshot, recorded in `Base_AsofDt`.

## Market rate

`Market_Rate` is the **average of the latest observation of each of 3 series** on/before the anchor date —
`MORTGAGE30US` (Freddie Mac PMMS), `OBMMIC30YF` (Optimal Blue), `MND_30Y` (Mortgage News Daily).

---

## Output columns

| Category | Column | Source / Notes |
|---|---|---|
| **Key** | `LoanNum` | `B2BPricingHist` — loan identifier (varchar 10) |
| **Key** | `AsofDt` | `B2BPricingHist` — month-end snapshot date |
| **Display (AsofDt)** | `FICOOrig` | `vwServLoanHist` at `AsofDt` |
| **Display (AsofDt)** | `PrinBal` | `vwServLoanHist` at `AsofDt` — current UPB |
| **Display (AsofDt)** | `IntRate` | `vwServLoanHist` at `AsofDt`, in % |
| **Display (AsofDt)** | `Market_Rate` | 3-series blend at `AsofDt`, in % |
| **Display (AsofDt)** | `InvestorCatgy` | `vwServLoanHist` at `AsofDt` — not a cohort input |
| **Display (AsofDt)** | `LoanAge` | `DATEDIFF(MONTH, OrigDt, AsofDt)` — not a cohort input |
| **Display + cohort key** | `RemainingTerm` | `vwServLoanHist.TermRemaining` at `AsofDt` (months) |
| **Display + cohort key** | `LoanType` | `vwServLoanHist` at `AsofDt` — raw value (normalized to CONV/FHA/VA/OTHERS at scoring) |
| **Transparency** | `Channel` | `B2BPricingHist` — Correspondent / Co-Issue / Consumer Direct |
| **Transparency** | `FundDt` | `B2BPricingHist` — funding date |
| **Transparency** | `LockDt` | `vwOrigLoan` — rate lock date (null for Co-Issue) |
| **Transparency** | `CohortBaseDt` | The date actually used to pull cohort attributes |
| **Transparency** | `CohortBaseDtType` | `'LockDt'` or `'FundDt'` — which date was used |
| **Transparency** | `Base_AsofDt` | Month-end snapshot the baseline attributes came from |
| **Baseline (cohort input)** | `FICOOrig_Base` | `vwServLoanHist` at `Base_AsofDt` |
| **Baseline (cohort input)** | `PrinBal_Base` | `vwServLoanHist` at `Base_AsofDt` |
| **Baseline (cohort input)** | `IntRate_Base` | `vwServLoanHist` at `Base_AsofDt`, in % |
| **Baseline (cohort input)** | `Market_Rate_Base` | 3-series blend at `CohortBaseDt`, in % |
| **Baseline (cohort input)** | `Market_Rate_BaseDt` | Average observation date behind `Market_Rate_Base` |
| **Baseline (cohort input)** | `RemainingTerm_Base` | `TermRemaining` at `Base_AsofDt`, rolled back to `CohortBaseDt` (`+ DATEDIFF(MONTH, CohortBaseDt, Base_AsofDt)`) |
| **Baseline (cohort input)** | `LoanType_Base` | `vwServLoanHist` at `Base_AsofDt` — raw value |
| **Baseline** | `InvestorCatgy_Base` | `vwServLoanHist` at `Base_AsofDt` — not a cohort input |
| **Baseline** | `LoanAge_Base` | `DATEDIFF(MONTH, OrigDt, CohortBaseDt)` — not a cohort input |
| **Cohort** | `Rule_Bucket` | `cohort_definitions.csv` — cohort rule string |
| **Cohort** | `Mean_CPR` | Predicted cohort mean CPR (%) |
| **Cohort** | `Std_CPR` | Cohort CPR standard deviation |
| **Cohort** | `Cohort_ID` | `YYYYMM_C{seq:03d}` — `YYYYMM` from `CohortBaseDt` |
| **Cohort** | `cohort_unique_ID` | `C{seq:03d}` — invariant across all snapshots |

Use the **Python (B2B)** kernel (env: `B2B`).


## 0. Parameters & Configuration

In [ ]:
# ── Database & target table ───────────────────────────────────────────────────
SERVER          = 'VMC2VMDBRADC01P'
DATABASE        = 'Analytics_Rpt'
TARGET_SCHEMA   = 'md'
TARGET_TABLE    = 'B2B_revamp_cohorts'
FULL_TABLE_NAME = f'[{DATABASE}].[{TARGET_SCHEMA}].[{TARGET_TABLE}]'
ODBC_DRIVER     = 'ODBC Driver 17 for SQL Server'

# ── Operational Mode ──────────────────────────────────────────────────────────
# 'APPEND'         : Add new month(s) to base table + assign cohorts for new loans
#                    Use this for routine monthly updates.
# 'COHORT_REFRESH' : Base table unchanged; clear ALL cohort columns + re-assign from scratch.
#                    Use this when cohort definitions change and all history needs to be re-scored.
# 'FULL_REBUILD'   : DROP + full rebuild of base table + assign cohorts for all rows.
#                    Use this for initial build or if base attributes need recalculation.
RUN_MODE = 'APPEND'

# ── APPEND mode: month-end date(s) to add ─────────────────────────────────────
# [] = auto-detect (all months in B2BPricingHist not yet in target table)
# Explicit: ['2026-07-31'] or ['2026-07-31', '2026-08-31']
NEW_MONTHS = ['2026-10-31']

# ── Source filter ─────────────────────────────────────────────────────────────
# Standard B2B business rule — exclude zero/null funded rows from B2BPricingHist
FUNDED_FILTER = True

# ── Cohort anchor date rule (by B2BPricingHist.Channel) ──────────────────────
# Correspondent  -> LockDt from ADM.ODS.vwOrigLoan (falls back to FundDt if NULL)
# Co-Issue       -> FundDt
# Consumer Direct-> FundDt
# Anything else  -> FundDt
LOCKDT_CHANNELS = ('Correspondent',)

# Baseline servicing attributes are read at EOMONTH(CohortBaseDt); if that month-end
# snapshot does not exist yet (common for Correspondent, where lock precedes boarding),
# cascade forward up to this many months to the first available snapshot.
MAX_BASE_FALLBACK_MONTHS = 3

# ── Market rate blend ─────────────────────────────────────────────────────────
# Market_Rate = average of the latest observation of each series on/before the anchor date
RATE_SERIES = ('MORTGAGE30US', 'OBMMIC30YF', 'MND_30Y')

# ── Derived clauses (do not edit) ─────────────────────────────────────────────
FUNDED_CLAUSE   = 'AND h.FundedLoanAmt > 0' if FUNDED_FILTER else ''
RATE_SERIES_SQL = "'" + "','".join(RATE_SERIES) + "'"
LOCK_CHANNEL_SQL = "'" + "','".join(LOCKDT_CHANNELS) + "'"

print(f"Target table     : {FULL_TABLE_NAME}")
print(f"Run mode         : {RUN_MODE}")
print(f"Funded filter    : {FUNDED_FILTER}")
print(f"LockDt channels  : {', '.join(LOCKDT_CHANNELS)}  (all other channels use FundDt)")
print(f"Base fallback    : EOMONTH(CohortBaseDt) + up to {MAX_BASE_FALLBACK_MONTHS} month(s)")
print(f"Rate series      : {', '.join(RATE_SERIES)}")
if RUN_MODE == 'APPEND':
    print(f"New months       : {NEW_MONTHS if NEW_MONTHS else '(auto-detect from B2BPricingHist)'}")


Target table     : [Analytics_Rpt].[md].[B2B_revamp_cohorts]
Run mode         : FULL_REBUILD
Funded filter    : True
LockDt channels  : Correspondent  (all other channels use FundDt)
Base fallback    : EOMONTH(CohortBaseDt) + up to 3 month(s)
Rate series      : MORTGAGE30US, OBMMIC30YF, MND_30Y


## 1. Imports & SQL Connection

In [2]:
import pyodbc
import pandas as pd
import numpy as np
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ── Analytics_Rpt connection — write + DDL access needed for CREATE TABLE ─────
conn_str = (
    f"DRIVER={{{ODBC_DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=Yes;"
)
try:
    cn.cursor()
    print("✅ Reusing existing Analytics_Rpt connection")
except:
    cn = pyodbc.connect(conn_str)
    cn.autocommit = True   # Required for DDL statements (DROP / CREATE TABLE)
    print(f"✅ Connected to {DATABASE} on {SERVER}")

✅ Connected to Analytics_Rpt on VMC2VMDBRADC01P


## 2. Source Coverage — B2BPricingHist Snapshot & FundDt Overview

Check total `LoanNum × AsofDt` pairs, unique loan count, and `FundDt` availability across all snapshot months.

In [ ]:
query_coverage = f"""
SELECT
    AsofDt,
    COUNT(DISTINCT LoanNum) AS LoanCount,
    COUNT(*)                AS NumRows,
    COUNT(FundDt)           AS FundDtCount
FROM Analytics_Rpt.md.B2BPricingHist h
WHERE 1 = 1 {FUNDED_CLAUSE}
GROUP BY AsofDt
ORDER BY AsofDt
"""
df_coverage = pd.read_sql(query_coverage, cn)
df_coverage['AsofDt'] = pd.to_datetime(df_coverage['AsofDt'])
total_pairs = df_coverage['NumRows'].sum()

print(f"B2BPricingHist date range     : {df_coverage['AsofDt'].min().date()}  →  {df_coverage['AsofDt'].max().date()}")
print(f"Total monthly snapshots       : {len(df_coverage)}")
print(f"Total LoanNum × AsofDt rows   : {total_pairs:,}")
print(f"(These {total_pairs:,} rows will be written to {FULL_TABLE_NAME})")
print()
print("Last 6 months snapshot summary:")
print(df_coverage.tail(6).to_string(index=False))

# ── Channel mix & LockDt availability — drives the cohort anchor date rule ────
query_channel = f"""
SELECT
    h.Channel,
    COUNT(DISTINCT h.LoanNum)                                            AS Loans,
    COUNT(DISTINCT CASE WHEN o.LockDt IS NOT NULL THEN h.LoanNum END)    AS With_LockDt,
    COUNT(DISTINCT CASE WHEN h.FundDt IS NOT NULL THEN h.LoanNum END)    AS With_FundDt,
    CASE WHEN h.Channel IN ({LOCK_CHANNEL_SQL}) THEN 'LockDt (fallback FundDt)'
         ELSE 'FundDt' END                                               AS Anchor_Rule
FROM Analytics_Rpt.md.B2BPricingHist h
LEFT JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
WHERE 1 = 1 {FUNDED_CLAUSE}
GROUP BY h.Channel
ORDER BY Loans DESC
"""
df_channel = pd.read_sql(query_channel, cn)
print("\nChannel mix & cohort anchor date rule:")
print(df_channel.to_string(index=False))


B2BPricingHist date range     : 2023-02-28  →  2026-09-30
Total monthly snapshots       : 44
Total LoanNum × AsofDt rows   : 12,258,573
(These 12,258,573 rows will be written to [Analytics_Rpt].[md].[B2B_revamp_cohorts])

Last 6 months snapshot summary:
    AsofDt  LoanCount  NumRows  FundDtCount
2026-04-30     363145   363145       363142
2026-05-31     378787   378787       378784
2026-06-30     386597   386597       386594
2026-07-31     395048   395048       395046
2026-08-31     392364   392364       392362
2026-09-30     407134   407134       407132

Channel mix & cohort anchor date rule:
        Channel  Loans  With_LockDt  With_FundDt              Anchor_Rule
       Co-Issue 279215            0       279215                   FundDt
  Correspondent 256230       256228       256225 LockDt (fallback FundDt)
Consumer Direct  31906        31905        31906                   FundDt


## 3. Data Quality — vwServLoanHist Join Coverage (AsofDt vs FundDt)

Verify join match percentage in `ADM.ODS.vwServLoanHist` for:
1. Current snapshot (`AsofDt`)
2. Baseline funding snapshot (`CAST(EOMONTH(FundDt) AS DATE)`)

In [ ]:
latest_dt = df_coverage['AsofDt'].max().strftime('%Y-%m-%d')

_offsets = list(range(MAX_BASE_FALLBACK_MONTHS + 1))

def _sql_coalesce(parts):
    """SQL Server rejects COALESCE with a single argument."""
    parts = list(parts)
    return parts[0] if len(parts) == 1 else "COALESCE(" + ", ".join(parts) + ")"

_cov_joins = "\n".join(
    f"LEFT JOIN ADM.ODS.vwServLoanHist sb{i}\n"
    f"    ON b.LoanNum = sb{i}.LoanNum AND sb{i}.AsOfDt = CAST(EOMONTH(b.CohortBaseDt, {i}) AS DATE)"
    for i in _offsets
)
_cov_cols = "\n".join(
    f"    ,SUM(CASE WHEN {_sql_coalesce(f'sb{j}.LoanNum' for j in range(i + 1))} "
    f"IS NOT NULL THEN 1 ELSE 0 END) AS Hit_M0_M{i}"
    for i in _offsets
)

query_join_check = f"""
WITH Base AS (
    SELECT DISTINCT
        h.LoanNum,
        h.AsofDt,
        h.Channel,
        CohortBaseDt = CASE WHEN h.Channel IN ({LOCK_CHANNEL_SQL}) AND o.LockDt IS NOT NULL
                            THEN CAST(o.LockDt AS DATE) ELSE CAST(h.FundDt AS DATE) END
    FROM Analytics_Rpt.md.B2BPricingHist h
    LEFT JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
    WHERE h.AsofDt = '{latest_dt}' {FUNDED_CLAUSE}
)
SELECT
     b.Channel
    ,COUNT(*)                                                     AS Loans
    ,SUM(CASE WHEN s_curr.LoanNum IS NOT NULL THEN 1 ELSE 0 END)  AS Hit_AsofDt
{_cov_cols}
FROM Base b
LEFT JOIN ADM.ODS.vwServLoanHist s_curr
    ON b.LoanNum = s_curr.LoanNum AND b.AsofDt = s_curr.AsOfDt
{_cov_joins}
GROUP BY b.Channel
ORDER BY Loans DESC
"""
df_join = pd.read_sql(query_join_check, cn)

print(f"vwServLoanHist join coverage for latest snapshot ({latest_dt}):")
_disp = df_join.copy()
for c in [c for c in _disp.columns if c.startswith('Hit_')]:
    _disp[c] = (_disp[c] / _disp['Loans'] * 100).map('{:.1f}%'.format)
_disp['Loans'] = _disp['Loans'].map('{:,}'.format)
print(_disp.to_string(index=False))
print()
print("Hit_AsofDt  = display attributes matched at AsofDt")
print("Hit_M0_M0   = baseline matched at EOMONTH(CohortBaseDt)")
print(f"Hit_M0_M{MAX_BASE_FALLBACK_MONTHS}   = baseline matched after cascading up to +{MAX_BASE_FALLBACK_MONTHS} months")
print()
print("Correspondent locks precede boarding, so M0 coverage is low by design — the")
print("cascade picks the first available servicing snapshot on/after the lock month.")


vwServLoanHist join coverage for latest snapshot (2026-09-30):
        Channel   Loans Hit_AsofDt Hit_M0_M0 Hit_M0_M1 Hit_M0_M2 Hit_M0_M3
       Co-Issue 202,556     100.0%     95.8%    100.0%    100.0%    100.0%
  Correspondent 181,480     100.0%     45.1%     89.2%     98.9%     99.9%
Consumer Direct  23,098     100.0%     82.3%    100.0%    100.0%    100.0%

Hit_AsofDt  = display attributes matched at AsofDt
Hit_M0_M0   = baseline matched at EOMONTH(CohortBaseDt)
Hit_M0_M3   = baseline matched after cascading up to +3 months

Correspondent locks precede boarding, so M0 coverage is low by design — the
cascade picks the first available servicing snapshot on/after the lock month.


## 4. Market Rate — 3-Series Blend Preview

`Market_Rate` is the **average of the latest observation of each of 3 series** on/before the anchor date:

| Series | Source |
|---|---|
| `MORTGAGE30US` | Freddie Mac PMMS 30-yr fixed (weekly) |
| `OBMMIC30YF` | Optimal Blue Mortgage Market Index 30-yr conforming (daily) |
| `MND_30Y` | Mortgage News Daily 30-yr (daily) |

For each anchor date the query takes `MAX(RateDt) <= AnchorDt` **per series**, then averages across
series. `Market_Rate_BaseDt` records the latest observation date among the available series used in the blend,
for transparency. All three series are stored as decimal fractions, so the blend is multiplied by 100 to express a percentage.


In [ ]:
# ── Per-series coverage ───────────────────────────────────────────────────────
query_series = f"""
SELECT Series,
       COUNT(*)  AS Obs,
       MIN(RateDt) AS MinRateDt,
       MAX(RateDt) AS MaxRateDt
FROM ADM.Ext.MarketRates
WHERE Series IN ({RATE_SERIES_SQL})
GROUP BY Series
ORDER BY Series
"""
df_series = pd.read_sql(query_series, cn)
print("Rate series coverage:")
print(df_series.to_string(index=False))

# ── Blended rate at recent month-ends ────────────────────────────────────────
query_blend_preview = f"""
WITH AnchorDates AS (
    SELECT TOP 12 CAST(EOMONTH(AsofDt) AS DATE) AS D
    FROM Analytics_Rpt.md.B2BPricingHist
    GROUP BY EOMONTH(AsofDt)
    ORDER BY EOMONTH(AsofDt) DESC
)
SELECT a.D                                                          AS AnchorDt,
       CAST(AVG(m.Rate) * 100 AS DECIMAL(8,4))                      AS Market_Rate,
       CAST(DATEADD(DAY, AVG(DATEDIFF(DAY, '1900-01-01', m.RateDt)),
                    '1900-01-01') AS DATE)                          AS Market_Rate_Dt,
       COUNT(*)                                                     AS SeriesUsed
FROM AnchorDates a
INNER JOIN ADM.Ext.MarketRates m
    ON  m.Series IN ({RATE_SERIES_SQL})
    AND m.RateDt = (SELECT MAX(x.RateDt) FROM ADM.Ext.MarketRates x
                    WHERE x.Series = m.Series AND x.RateDt <= a.D)
GROUP BY a.D
ORDER BY a.D DESC
"""
df_blend = pd.read_sql(query_blend_preview, cn)
print(f"\nBlended {len(RATE_SERIES)}-series rate at the last 12 month-ends:")
print(df_blend.to_string(index=False))
print()
print("SeriesUsed < 3 means a series had not started yet at that anchor date (MND_30Y begins 2022-12).")


Rate series coverage:
      Series  Obs  MinRateDt  MaxRateDt
     MND_30Y  952 2022-12-01 2026-10-13
MORTGAGE30US 1397 2000-01-07 2026-10-08
  OBMMIC30YF 2442 2017-01-03 2026-10-09

Blended 3-series rate at the last 12 month-ends:
  AnchorDt  Market_Rate Market_Rate_Dt  SeriesUsed
2026-09-30       7.3386     2026-09-28           3
2026-08-31       6.7496     2026-08-29           3
2026-07-31       6.7353     2026-07-30           3
2026-06-30       6.4873     2026-06-28           3
2026-05-31       6.5093     2026-05-28           3
2026-04-30       6.3543     2026-04-30           3
2026-03-31       6.4003     2026-03-29           3
2026-02-28       5.9570     2026-02-26           3
2026-01-31       6.1086     2026-01-29           3
2025-12-31       6.1696     2025-12-31           3
2025-11-30       6.1980     2025-11-26           3
2025-10-31       6.2020     2025-10-30           3

SeriesUsed < 3 means a series had not started yet at that anchor date (MND_30Y begins 2022-12).


## 5. SQL Definition — Anchor Date Resolution & Dual Join Strategy

The build runs as one batch with two helper temp tables followed by the main population query.

**Step 1 — `#LoanBase` (one row per `LoanNum`)** resolves the cohort anchor date:

| `Channel` | `CohortBaseDt` | `CohortBaseDtType` |
|---|---|---|
| `Correspondent` | `ISNULL(vwOrigLoan.LockDt, FundDt)` | `LockDt` (or `FundDt` if lock missing) |
| `Co-Issue` | `FundDt` | `FundDt` |
| `Consumer Direct` | `FundDt` | `FundDt` |
| *any other* | `FundDt` | `FundDt` |

**Step 2 — `#RateLookup`** computes the blended 3-series `Market_Rate` for every distinct anchor date
(both `CohortBaseDt` values and month-end `AsofDt` values) using the latest observation per series
on/before that date.

**Step 3 — main query** joins:
- **Display attributes (as of `AsofDt`)** — `vwServLoanHist` on `LoanNum + AsofDt`, `#RateLookup` on `AsofDt`.
- **Baseline attributes (as of `CohortBaseDt`)** — `vwServLoanHist` cascading across
  `EOMONTH(CohortBaseDt, 0..N)` taking the first available snapshot, and `#RateLookup` on `CohortBaseDt`.

Baseline columns use a neutral `_Base` suffix since the anchor is now `LockDt` **or** `FundDt`
depending on channel.


In [ ]:
_offsets = list(range(MAX_BASE_FALLBACK_MONTHS + 1))

def _coalesce(template):
    """COALESCE across the cascading baseline snapshot aliases sb0..sbN."""
    return "COALESCE(" + ", ".join(template.format(i=i) for i in _offsets) + ")"

_base_joins = "\n".join(
    f"LEFT JOIN ADM.ODS.vwServLoanHist sb{i}\n"
    f"    ON  lb.LoanNum  = sb{i}.LoanNum\n"
    f"    AND sb{i}.AsOfDt = CAST(EOMONTH(lb.CohortBaseDt, {i}) AS DATE)"
    for i in _offsets
)

# Columns written to the target table, in order
TARGET_COLS = [
    'LoanNum', 'AsofDt',
    'FICOOrig', 'PrinBal', 'IntRate', 'Market_Rate', 'InvestorCatgy', 'LoanAge',
    'RemainingTerm', 'LoanType',
    'Channel', 'FundDt', 'LockDt', 'CohortBaseDt', 'CohortBaseDtType', 'Base_AsofDt',
    'FICOOrig_Base', 'PrinBal_Base', 'IntRate_Base', 'Market_Rate_Base',
    'Market_Rate_BaseDt', 'InvestorCatgy_Base', 'LoanAge_Base',
    'RemainingTerm_Base', 'LoanType_Base',
]

# ── Helper temp tables: anchor-date resolution + blended rate lookup ──────────
def build_prep_sql(month=None):
    month_clause = f"AND h.AsofDt = '{month}'" if month else ""
    return f"""
IF OBJECT_ID('tempdb..#LoanBase')   IS NOT NULL DROP TABLE #LoanBase;
IF OBJECT_ID('tempdb..#RateLookup') IS NOT NULL DROP TABLE #RateLookup;

-- 1. One row per loan: channel, FundDt, LockDt and the resolved cohort anchor date
SELECT
    h.LoanNum,
    Channel = MAX(h.Channel),
    FundDt  = CAST(MAX(h.FundDt)  AS DATE),
    LockDt  = CAST(MAX(o.LockDt)  AS DATE)
INTO #LoanBase
FROM Analytics_Rpt.md.B2BPricingHist h
LEFT JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
WHERE 1 = 1 {FUNDED_CLAUSE} {month_clause}
GROUP BY h.LoanNum;

ALTER TABLE #LoanBase ADD CohortBaseDt DATE NULL, CohortBaseDtType VARCHAR(10) NULL;

UPDATE #LoanBase
SET CohortBaseDt     = CASE WHEN Channel IN ({LOCK_CHANNEL_SQL}) AND LockDt IS NOT NULL
                            THEN LockDt ELSE FundDt END,
    CohortBaseDtType = CASE WHEN Channel IN ({LOCK_CHANNEL_SQL}) AND LockDt IS NOT NULL
                            THEN 'LockDt' ELSE 'FundDt' END;

CREATE CLUSTERED INDEX IX_LoanBase ON #LoanBase (LoanNum);

-- 2. Blended {len(RATE_SERIES)}-series market rate for every distinct anchor date
--    (latest observation per series on/before the anchor, then averaged across series)
--    Market_RateDt is the latest RateDt among the available series used in the blend.
WITH AnchorDates AS (
    SELECT CohortBaseDt AS D FROM #LoanBase WHERE CohortBaseDt IS NOT NULL
    UNION
    SELECT CAST(AsofDt AS DATE) FROM Analytics_Rpt.md.B2BPricingHist h
    WHERE 1 = 1 {FUNDED_CLAUSE} {month_clause}
)
SELECT
    RateAnchorDt  = a.D,
    Market_Rate   = CAST(AVG(m.Rate) * 100 AS DECIMAL(8,4)),
    Market_RateDt = CAST(MAX(m.RateDt) AS DATE)
INTO #RateLookup
FROM AnchorDates a
INNER JOIN ADM.Ext.MarketRates m
    ON  m.Series IN ({RATE_SERIES_SQL})
    AND m.RateDt = (SELECT MAX(x.RateDt) FROM ADM.Ext.MarketRates x
                    WHERE x.Series = m.Series AND x.RateDt <= a.D)
GROUP BY a.D;

CREATE CLUSTERED INDEX IX_RateLookup ON #RateLookup (RateAnchorDt);
"""

# Remaining term rolled back to CohortBaseDt: snapshot sb{i} is i month-ends after CohortBaseDt's month
_rem_term_base = "COALESCE(" + ", ".join(f"sb{i}.TermRemaining + {i}" for i in _offsets) + ")"

# ── Shared SELECT body (identical for FULL_REBUILD and APPEND) ────────────────
def build_select_body(month=None, into_clause=""):
    month_clause = f"AND h.AsofDt = '{month}'" if month else ""
    return f"""
SELECT
    b.LoanNum,
    b.AsofDt,
    -- ── Display attributes, as of AsofDt ─────────────────────────────────────
    s_curr.FICOOrig,
    s_curr.PrinBal,
    IntRate       = CAST(s_curr.IntRate * 100 AS DECIMAL(8,4)),
    Market_Rate   = r_curr.Market_Rate,
    s_curr.InvestorCatgy,
    LoanAge       = DATEDIFF(MONTH, s_curr.OrigDt, s_curr.AsOfDt),
    RemainingTerm = s_curr.TermRemaining,
    s_curr.LoanType,
    -- ── Anchor-date transparency ─────────────────────────────────────────────
    lb.Channel,
    lb.FundDt,
    lb.LockDt,
    lb.CohortBaseDt,
    lb.CohortBaseDtType,
    Base_AsofDt   = {_coalesce('sb{i}.AsOfDt')},
    -- ── Baseline attributes, as of CohortBaseDt (drive cohort assignment) ────
    FICOOrig_Base      = {_coalesce('sb{i}.FICOOrig')},
    PrinBal_Base       = {_coalesce('sb{i}.PrinBal')},
    IntRate_Base       = CAST({_coalesce('sb{i}.IntRate')} * 100 AS DECIMAL(8,4)),
    Market_Rate_Base   = r_base.Market_Rate,
    Market_Rate_BaseDt = r_base.Market_RateDt,
    InvestorCatgy_Base = {_coalesce('sb{i}.InvestorCatgy')},
    -- Correspondent locks can precede origination; floor the resulting age at 0
    LoanAge_Base       = CASE WHEN DATEDIFF(MONTH, {_coalesce('sb{i}.OrigDt')}, lb.CohortBaseDt) < 0
                              THEN 0
                              ELSE DATEDIFF(MONTH, {_coalesce('sb{i}.OrigDt')}, lb.CohortBaseDt) END,
    RemainingTerm_Base = {_rem_term_base},
    LoanType_Base      = {_coalesce('sb{i}.LoanType')}
{into_clause}
FROM (
    SELECT DISTINCT h.LoanNum, h.AsofDt
    FROM Analytics_Rpt.md.B2BPricingHist h
    WHERE 1 = 1 {FUNDED_CLAUSE} {month_clause}
) b
JOIN #LoanBase lb
    ON b.LoanNum = lb.LoanNum
LEFT JOIN ADM.ODS.vwServLoanHist s_curr
    ON  b.LoanNum = s_curr.LoanNum
    AND b.AsofDt  = s_curr.AsOfDt
{_base_joins}
LEFT JOIN #RateLookup r_curr
    ON r_curr.RateAnchorDt = b.AsofDt
LEFT JOIN #RateLookup r_base
    ON r_base.RateAnchorDt = lb.CohortBaseDt;
"""

sql_drop = f"""
IF OBJECT_ID('{FULL_TABLE_NAME}', 'U') IS NOT NULL
    DROP TABLE {FULL_TABLE_NAME};
"""

sql_full_build = build_prep_sql() + build_select_body(into_clause=f"INTO {FULL_TABLE_NAME}")

sql_add_index = f"""
CREATE CLUSTERED INDEX IX_{TARGET_TABLE}_LoanNum_AsofDt
ON {FULL_TABLE_NAME} (LoanNum, AsofDt);
"""

print(f"Current RUN_MODE : {RUN_MODE}")
print(f"Target columns   : {len(TARGET_COLS)} + 5 cohort columns added in Section 9")
if RUN_MODE == 'FULL_REBUILD':
    print("\n── DROP ─────────────────────────────────────────────────────────────────")
    print(sql_drop)
    print("── PREP + SELECT INTO ───────────────────────────────────────────────────")
    print(sql_full_build)
    print("── CREATE CLUSTERED INDEX ───────────────────────────────────────────────")
    print(sql_add_index)
else:
    print(f"SQL builders defined for mode '{RUN_MODE}'.")


Current RUN_MODE : FULL_REBUILD
Target columns   : 25 + 5 cohort columns added in Section 9

── DROP ─────────────────────────────────────────────────────────────────

IF OBJECT_ID('[Analytics_Rpt].[md].[B2B_revamp_cohorts]', 'U') IS NOT NULL
    DROP TABLE [Analytics_Rpt].[md].[B2B_revamp_cohorts];

── PREP + SELECT INTO ───────────────────────────────────────────────────

IF OBJECT_ID('tempdb..#LoanBase')   IS NOT NULL DROP TABLE #LoanBase;
IF OBJECT_ID('tempdb..#RateLookup') IS NOT NULL DROP TABLE #RateLookup;

-- 1. One row per loan: channel, FundDt, LockDt and the resolved cohort anchor date
SELECT
    h.LoanNum,
    Channel = MAX(h.Channel),
    FundDt  = CAST(MAX(h.FundDt)  AS DATE),
    LockDt  = CAST(MAX(o.LockDt)  AS DATE)
INTO #LoanBase
FROM Analytics_Rpt.md.B2BPricingHist2 h
LEFT JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
WHERE 1 = 1 AND h.FundedLoanAmt > 0 
GROUP BY h.LoanNum;

ALTER TABLE #LoanBase ADD CohortBaseDt DATE NULL, CohortBaseDtType VARCHAR(10) NULL;

UP

## 6. Create & Populate Base Table

Behaviour is controlled by `RUN_MODE` set in Section 0:

| `RUN_MODE` | Base table action | Typical use |
|---|---|---|
| `APPEND` | INSERT new month(s) only; existing rows untouched | **Routine monthly update** |
| `COHORT_REFRESH` | Skipped — base table unchanged | Re-score cohorts after definition change |
| `FULL_REBUILD` | DROP + SELECT INTO + clustered index | First-time build or full reset |

In [7]:
def _drain_results(cursor):
    while True:
        if cursor.description is not None:
            cursor.fetchall()
        if not cursor.nextset():
            break


if 'cursor' in globals():
    try:
        _drain_results(cursor)
        cursor.close()
    except pyodbc.Error:
        pass

cursor = cn.cursor()

# ── COHORT_REFRESH: base table untouched ─────────────────────────────────────
if RUN_MODE == 'COHORT_REFRESH':
    print(f"RUN_MODE = COHORT_REFRESH")
    print(f"  Skipping base table — {FULL_TABLE_NAME} is unchanged.")
    print(f"  All cohort columns will be cleared and re-assigned in Section 10.")
    MONTHS_ADDED = []

# ── FULL_REBUILD: DROP + PREP + SELECT INTO + clustered index ────────────────
elif RUN_MODE == 'FULL_REBUILD':
    print(f"RUN_MODE = FULL_REBUILD")

    print(f"Step 1: Dropping {FULL_TABLE_NAME} (if exists) ...")
    cursor.execute(sql_drop)
    print("        Done.")

    print(f"Step 2: Building {FULL_TABLE_NAME} ...")
    print(f"        #LoanBase (anchor dates) → #RateLookup ({len(RATE_SERIES)}-series blend)")
    print(f"        → vwServLoanHist [AsofDt + CohortBaseDt cascade M0..M{MAX_BASE_FALLBACK_MONTHS}]")
    t0 = time.time()
    cursor.execute(sql_full_build)
    _drain_results(cursor)
    elapsed      = time.time() - t0
    # cursor.rowcount reflects only the first statement of the batch, so count the table directly
    rows_written = pd.read_sql(f"SELECT COUNT(*) AS n FROM {FULL_TABLE_NAME}", cn).at[0, 'n']
    print(f"        ✅ {rows_written:,} rows written  ({elapsed:.1f}s)")

    print(f"Step 3: Creating clustered index on (LoanNum, AsofDt) ...")
    cursor.execute(sql_add_index)
    _drain_results(cursor)
    print("        ✅ Index created.")

    MONTHS_ADDED = df_coverage['AsofDt'].dt.strftime('%Y-%m-%d').tolist()
    print(f"\n✅ {FULL_TABLE_NAME} is ready.")

# ── APPEND: detect & insert new months only ───────────────────────────────────
elif RUN_MODE == 'APPEND':
    print(f"RUN_MODE = APPEND")

    # Guard: table must already exist
    cursor.execute(f"""
        SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_CATALOG = '{DATABASE}'
          AND TABLE_SCHEMA   = '{TARGET_SCHEMA}'
          AND TABLE_NAME     = '{TARGET_TABLE}'
    """)
    if cursor.fetchone()[0] == 0:
        raise RuntimeError(
            f"{FULL_TABLE_NAME} does not exist. Set RUN_MODE = 'FULL_REBUILD' to create it first."
        )

    df_existing = pd.read_sql(
        f"SELECT DISTINCT CONVERT(VARCHAR(10), AsofDt, 120) AS AsofDt FROM {FULL_TABLE_NAME}",
        cn
    )
    existing_months = set(df_existing['AsofDt'].tolist())

    if NEW_MONTHS:
        already_in    = [m for m in NEW_MONTHS if m in existing_months]
        months_to_add = sorted(m for m in NEW_MONTHS if m not in existing_months)
        if already_in:
            print(f"  Already in table (skipping)  : {already_in}")
    else:
        all_b2b_months = set(df_coverage['AsofDt'].dt.strftime('%Y-%m-%d').tolist())
        months_to_add  = sorted(all_b2b_months - existing_months)

    if not months_to_add:
        print("  ✅ No new months detected — base table is already up to date.")
        MONTHS_ADDED = []
    else:
        print(f"  New month(s) to insert: {months_to_add}")
        MONTHS_ADDED = []

        for month in months_to_add:
            t0   = time.time()
            _sql = (
                build_prep_sql(month)
                + f"\nINSERT INTO {FULL_TABLE_NAME} ({', '.join(TARGET_COLS)})\n"
                + build_select_body(month)
            )
            cursor.execute(_sql)
            rows = cursor.rowcount
            _drain_results(cursor)
            elapsed = time.time() - t0
            print(f"  ✅ {month}: {rows:,} rows inserted  ({elapsed:.1f}s)")
            MONTHS_ADDED.append(month)

        print(f"\n✅ {len(MONTHS_ADDED)} new month(s) added to {FULL_TABLE_NAME}.")

cursor.close()

RUN_MODE = FULL_REBUILD
Step 1: Dropping [Analytics_Rpt].[md].[B2B_revamp_cohorts] (if exists) ...
        Done.
Step 2: Building [Analytics_Rpt].[md].[B2B_revamp_cohorts] ...
        #LoanBase (anchor dates) → #RateLookup (3-series blend)
        → vwServLoanHist [AsofDt + CohortBaseDt cascade M0..M3]
        ✅ 12,258,573 rows written  (190.1s)
Step 3: Creating clustered index on (LoanNum, AsofDt) ...
        ✅ Index created.

✅ [Analytics_Rpt].[md].[B2B_revamp_cohorts] is ready.


## 7. Validation — Base Table Audit (Row Counts & Null Audit)

Post-write verification of row counts, key columns null audit (both current `AsofDt` and baseline `FundDt`), and negative `LoanAge` sanity check.

In [ ]:
# ── Check 1: Row count vs expected ────────────────────────────────────────────
query_rowcount = f"""
SELECT
    COUNT(*) AS TotalRows,
    COUNT(DISTINCT CONCAT(LoanNum, '|', CAST(AsofDt AS VARCHAR))) AS DistinctPairs,
    COUNT(DISTINCT LoanNum)                                       AS DistinctLoans
FROM {FULL_TABLE_NAME}
"""
df_rc = pd.read_sql(query_rowcount, cn)
actual_rows     = df_rc.at[0, 'TotalRows']
distinct_pairs  = df_rc.at[0, 'DistinctPairs']
distinct_loans  = df_rc.at[0, 'DistinctLoans']

print("CHECK 1 — Row count")
print(f"  Expected (from B2BPricingHist)  : {total_pairs:,}")
print(f"  Actual (in target table)        : {actual_rows:,}")
print(f"  Distinct LoanNum × AsofDt pairs : {distinct_pairs:,}")
print(f"  Distinct loans                  : {distinct_loans:,}")
match_ok = actual_rows == total_pairs
print(f"  Match                           : {'✅ PASS' if match_ok else '❌ MISMATCH'}")

# ── Check 2: Null audit — AsofDt (display) vs Base (cohort input) ─────────────
query_nulls = f"""
SELECT
    COUNT(*)                                                         AS TotalRows,
    SUM(CASE WHEN FICOOrig             IS NULL THEN 1 ELSE 0 END)    AS FICOOrig_Null,
    SUM(CASE WHEN PrinBal              IS NULL THEN 1 ELSE 0 END)    AS PrinBal_Null,
    SUM(CASE WHEN IntRate              IS NULL THEN 1 ELSE 0 END)    AS IntRate_Null,
    SUM(CASE WHEN Market_Rate          IS NULL THEN 1 ELSE 0 END)    AS Market_Rate_Null,
    SUM(CASE WHEN InvestorCatgy        IS NULL THEN 1 ELSE 0 END)    AS InvestorCatgy_Null,
    SUM(CASE WHEN LoanAge              IS NULL THEN 1 ELSE 0 END)    AS LoanAge_Null,
    SUM(CASE WHEN RemainingTerm        IS NULL THEN 1 ELSE 0 END)    AS RemainingTerm_Null,
    SUM(CASE WHEN LoanType             IS NULL THEN 1 ELSE 0 END)    AS LoanType_Null,
    SUM(CASE WHEN FICOOrig_Base        IS NULL THEN 1 ELSE 0 END)    AS FICOOrig_Base_Null,
    SUM(CASE WHEN PrinBal_Base         IS NULL THEN 1 ELSE 0 END)    AS PrinBal_Base_Null,
    SUM(CASE WHEN IntRate_Base         IS NULL THEN 1 ELSE 0 END)    AS IntRate_Base_Null,
    SUM(CASE WHEN Market_Rate_Base     IS NULL THEN 1 ELSE 0 END)    AS Market_Rate_Base_Null,
    SUM(CASE WHEN InvestorCatgy_Base   IS NULL THEN 1 ELSE 0 END)    AS InvestorCatgy_Base_Null,
    SUM(CASE WHEN LoanAge_Base         IS NULL THEN 1 ELSE 0 END)    AS LoanAge_Base_Null,
    SUM(CASE WHEN RemainingTerm_Base   IS NULL THEN 1 ELSE 0 END)    AS RemainingTerm_Base_Null,
    SUM(CASE WHEN LoanType_Base        IS NULL THEN 1 ELSE 0 END)    AS LoanType_Base_Null,
    SUM(CASE WHEN CohortBaseDt         IS NULL THEN 1 ELSE 0 END)    AS CohortBaseDt_Null,
    SUM(CASE WHEN Base_AsofDt          IS NULL THEN 1 ELSE 0 END)    AS Base_AsofDt_Null,
    SUM(CASE WHEN LoanAge              < 0    THEN 1 ELSE 0 END)     AS LoanAge_Negative,
    SUM(CASE WHEN LoanAge_Base         < 0    THEN 1 ELSE 0 END)     AS LoanAge_Base_Negative,
    SUM(CASE WHEN RemainingTerm        < 0    THEN 1 ELSE 0 END)     AS RemainingTerm_Negative,
    SUM(CASE WHEN RemainingTerm_Base   < 0    THEN 1 ELSE 0 END)     AS RemainingTerm_Base_Negative
FROM {FULL_TABLE_NAME}
"""
df_nulls = pd.read_sql(query_nulls, cn)
total = df_nulls.at[0, 'TotalRows']

print("\nCHECK 2 — Null audit (display as-of AsofDt  vs  baseline as-of CohortBaseDt)")
print(f"{'Attribute':<16} | {'AsofDt NULLs':>14} {'(%)':>8} | {'Base NULLs':>14} {'(%)':>8}")
print("─" * 70)
for c in ['FICOOrig', 'PrinBal', 'IntRate', 'Market_Rate', 'InvestorCatgy', 'LoanAge', 'RemainingTerm', 'LoanType']:
    n_curr = df_nulls.at[0, f'{c}_Null']
    n_base = df_nulls.at[0, f'{c}_Base_Null']
    print(f"{c:<16} | {n_curr:>14,} {n_curr/total*100:>7.2f}% | {n_base:>14,} {n_base/total*100:>7.2f}%")

print()
for c in ['CohortBaseDt', 'Base_AsofDt']:
    n = df_nulls.at[0, f'{c}_Null']
    print(f"{c:<16} NULLs: {n:>10,}  ({n/total*100:.2f}%)")

neg_curr = df_nulls.at[0, 'LoanAge_Negative']
neg_base = df_nulls.at[0, 'LoanAge_Base_Negative']
print(f"\nNegative LoanAge (AsofDt) : {neg_curr:,}  {'✅ PASS' if neg_curr == 0 else '⚠️  check OrigDt'}")
print(f"Negative LoanAge_Base     : {neg_base:,}  "
      f"{'✅ PASS' if neg_base == 0 else '⚠️  lock precedes origination for some loans (falls in 0-2yr band)'}")
neg_rt_curr = df_nulls.at[0, 'RemainingTerm_Negative']
neg_rt_base = df_nulls.at[0, 'RemainingTerm_Base_Negative']
print(f"Negative RemainingTerm (AsofDt) : {neg_rt_curr:,}  {'✅ PASS' if neg_rt_curr == 0 else '⚠️  maps to Missing term band'}")
print(f"Negative RemainingTerm_Base     : {neg_rt_base:,}  {'✅ PASS' if neg_rt_base == 0 else '⚠️  maps to Missing term band'}")

# ── Check 3: Anchor-date rule applied as expected ────────────────────────────
query_anchor = f"""
SELECT
    Channel,
    CohortBaseDtType,
    COUNT(DISTINCT LoanNum)                                                AS Loans,
    AVG(CAST(DATEDIFF(DAY, CohortBaseDt, FundDt) AS FLOAT))                AS Avg_Days_Base_To_Fund,
    AVG(CAST(DATEDIFF(MONTH, CohortBaseDt, Base_AsofDt) AS FLOAT))         AS Avg_Fallback_Months
FROM {FULL_TABLE_NAME}
GROUP BY Channel, CohortBaseDtType
ORDER BY Loans DESC
"""
df_anchor = pd.read_sql(query_anchor, cn)
df_anchor['Loans']                 = df_anchor['Loans'].map('{:,}'.format)
df_anchor['Avg_Days_Base_To_Fund'] = df_anchor['Avg_Days_Base_To_Fund'].map(
    lambda x: f'{x:.1f}' if pd.notna(x) else 'N/A')
df_anchor['Avg_Fallback_Months']   = df_anchor['Avg_Fallback_Months'].map(
    lambda x: f'{x:.2f}' if pd.notna(x) else 'N/A')
print("\nCHECK 3 — Anchor date rule by channel:")
print(df_anchor.to_string(index=False))


CHECK 1 — Row count
  Expected (from B2BPricingHist22222222)  : 12,258,573
  Actual (in target table)        : 12,258,573
  Distinct LoanNum × AsofDt pairs : 12,258,573
  Distinct loans                  : 567,351
  Match                           : ✅ PASS

CHECK 2 — Null audit (display as-of AsofDt  vs  baseline as-of CohortBaseDt)
Attribute        |   AsofDt NULLs      (%) |     Base NULLs      (%)
──────────────────────────────────────────────────────────────────────
FICOOrig         |         32,837    0.27% |         39,293    0.32%
PrinBal          |              0    0.00% |          6,456    0.05%
IntRate          |              0    0.00% |          6,456    0.05%
Market_Rate      |              0    0.00% |              0    0.00%
InvestorCatgy    |              0    0.00% |          6,456    0.05%
LoanAge          |              3    0.00% |          6,459    0.05%
RemainingTerm    |              0    0.00% |          6,456    0.05%
LoanType         |              0    0.00% 

## 8. Cohort Assignment — Load Artifacts & Algorithm

Loads the schema-version-4 artifacts from `cohorts_band/` (see `cohort_README.md`):
`cohort_bands_config.json`, `prinbal_map.csv`, `spread_map.csv`, `cohort_definitions.csv`.

Cohort keys: `LoanType`, `RemainingTermBand`, `FICOOrigBand`, `PrinBalGroup`, `RateSpreadGroup`.
There is no FICO map and no term map — FICO and RemainingTerm bands are fixed and never merged.


In [9]:
COHORT_DIR = os.getcwd()   # cohorts_band folder

# Load JSON band config (schema version 4); "-inf"/"inf" are strings -> float infinities
with open(os.path.join(COHORT_DIR, 'cohort_bands_config.json'), 'r') as f:
    cfg = json.load(f)
assert cfg['schema_version'] == 4, f"Unexpected cohort schema version: {cfg['schema_version']}"

def _edges(values):
    return [-np.inf if v == '-inf' else np.inf if v == 'inf' else float(v) for v in values]

# Load cohort lookup tables as strings (no NA parsing) so labels like 'OTHERS' stay intact
def _read_map(name):
    return pd.read_csv(os.path.join(COHORT_DIR, name), dtype=str, keep_default_na=False)

prinbal_map = _read_map('prinbal_map.csv')
spread_map  = _read_map('spread_map.csv')
cohort_defs = _read_map('cohort_definitions.csv')

# Stats columns come in as strings; blank Mean_CPR = extrapolated segment -> NaN
for _c in ['N', 'UPB', 'Mean_CPR', 'Std_CPR', 'SE_Mean', 'CV']:
    cohort_defs[_c] = pd.to_numeric(cohort_defs[_c].str.replace(',', ''), errors='coerce')

COHORT_KEYS = cfg['cohort_keys']   # LoanType, RemainingTermBand, FICOOrigBand, PrinBalGroup, RateSpreadGroup

# Build stable Rule → sequential-number mapping for Cohort_ID construction
rule_to_seq = (
    cohort_defs[COHORT_KEYS + ['Rule']]
    .drop_duplicates(subset=COHORT_KEYS)
    .sort_values(COHORT_KEYS)
    .reset_index(drop=True)
)
rule_to_seq['RuleSeq'] = (rule_to_seq.index + 1).astype(str).str.zfill(3)

def normalize_loan_type(values):
    v = values.astype('string').str.strip().str.upper().replace({'CONV-PMI': 'CONV'})
    return v.where(v.isin(['CONV', 'FHA', 'VA']), 'OTHERS').astype(object)

def _band(values, edge_key, label_key):
    return pd.cut(values, _edges(cfg[edge_key]), labels=cfg[label_key], include_lowest=True).astype(object)

def assign_cohorts(df_in):
    """
    Apply the schema-v4 cohort assignment (cohort_README.md) to a loan DataFrame.
    Required columns: LoanType, RemainingTerm, FICOOrig, PrinBal, IntRate, Market_Rate
    Rows with missing PrinBal / IntRate / Market_Rate stay unmatched (Rule_Bucket = NaN).
    """
    df = df_in.copy()

    # Step 1: Normalize LoanType -> CONV / FHA / VA / OTHERS
    df['LoanType'] = normalize_loan_type(df['LoanType'])

    # Step 2: Bands
    term = pd.to_numeric(df['RemainingTerm'], errors='coerce')
    term = term.where(term >= 0)
    df['RemainingTermBand'] = _band(term, 'remainingterm_edges', 'remainingterm_labels').fillna(cfg['remainingterm_missing_label'])

    fico = pd.to_numeric(df['FICOOrig'], errors='coerce')
    fico = fico.where((fico > 300) & (fico < 850))      # sentinel / missing -> default band
    df['FICOOrigBand'] = _band(fico, 'ficoorig_edges', 'ficoorig_labels').fillna(cfg['ficoorig_missing_label'])

    df['RateSpread']     = df['IntRate'] - df['Market_Rate']
    df['PrinBalBand']    = _band(df['PrinBal'], 'prinbal_edges', 'prinbal_labels')
    df['RateSpreadBand'] = _band(df['RateSpread'], 'spread_edges', 'spread_labels')

    # Step 3: Join lookups -> PrinBalGroup -> RateSpreadGroup -> cohort statistics
    df = df.merge(prinbal_map, on=['LoanType', 'RemainingTermBand', 'FICOOrigBand', 'PrinBalBand'],
                  how='left', validate='many_to_one')
    df = df.merge(spread_map, on=['LoanType', 'RemainingTermBand', 'FICOOrigBand', 'PrinBalGroup', 'RateSpreadBand'],
                  how='left', validate='many_to_one')
    df = df.merge(cohort_defs[COHORT_KEYS + ['Rule', 'Mean_CPR', 'Std_CPR']], on=COHORT_KEYS,
                  how='left', validate='many_to_one')
    df.rename(columns={'Rule': 'Rule_Bucket'}, inplace=True)

    df = df.merge(rule_to_seq[COHORT_KEYS + ['RuleSeq']], on=COHORT_KEYS, how='left', validate='many_to_one')
    return df

print(f"✅ Loaded cohort config (schema v{cfg['schema_version']}) & lookup tables "
      f"({len(rule_to_seq)} unique cohorts, {int(cohort_defs['Mean_CPR'].notna().sum())} populated).")
print(f"   prinbal_map rows: {len(prinbal_map)} | spread_map rows: {len(spread_map)} | cohort keys: {COHORT_KEYS}")
print("✅ assign_cohorts() defined.")


✅ Loaded cohort config (schema v4) & lookup tables (116 unique cohorts, 109 populated).
   prinbal_map rows: 96 | spread_map rows: 648 | cohort keys: ['LoanType', 'RemainingTermBand', 'FICOOrigBand', 'PrinBalGroup', 'RateSpreadGroup']
✅ assign_cohorts() defined.


## 9. Add Cohort Columns & Create Staging Table

Adds `Rule_Bucket`, `Mean_CPR`, `Std_CPR`, `Cohort_ID`, and `cohort_unique_ID` to the main table.
Creates a staging table `[Analytics_Rpt].[md].[B2B_cohort_funddt_staging]` keyed by `LoanNum`.

In [10]:
cursor = cn.cursor()

NEW_COLS = [
    ('Rule_Bucket',      'NVARCHAR(500)'),
    ('Mean_CPR',         'FLOAT'),
    ('Std_CPR',          'FLOAT'),
    ('Cohort_ID',        'VARCHAR(50)'),
    ('cohort_unique_ID', 'VARCHAR(20)'),
]
print("Adding cohort columns to main table (idempotent):")
for col_name, col_type in NEW_COLS:
    try:
        cursor.execute(f"ALTER TABLE {FULL_TABLE_NAME} ADD {col_name} {col_type} NULL")
        print(f"  ✅ {col_name} {col_type}")
    except Exception as e:
        if 'already' in str(e).lower() or 'more than once' in str(e).lower():
            print(f"  ℹ️  {col_name} already exists — skipping ALTER")
        else:
            raise

STAGING_TABLE = f'[{DATABASE}].[{TARGET_SCHEMA}].[B2B_cohort_funddt_staging]'
cursor.execute(f"""
IF OBJECT_ID('{STAGING_TABLE}', 'U') IS NOT NULL
    DROP TABLE {STAGING_TABLE};
""")
cursor.execute(f"""
CREATE TABLE {STAGING_TABLE} (
    LoanNum          VARCHAR(10)   NOT NULL,
    Rule_Bucket      NVARCHAR(500) NULL,
    Mean_CPR         FLOAT         NULL,
    Std_CPR          FLOAT         NULL,
    Cohort_ID        VARCHAR(50)   NULL,
    cohort_unique_ID VARCHAR(20)   NULL
);
""")
print(f"\n✅ Staging table created: {STAGING_TABLE}")
cursor.close()

Adding cohort columns to main table (idempotent):
  ✅ Rule_Bucket NVARCHAR(500)
  ✅ Mean_CPR FLOAT
  ✅ Std_CPR FLOAT
  ✅ Cohort_ID VARCHAR(50)
  ✅ cohort_unique_ID VARCHAR(20)

✅ Staging table created: [Analytics_Rpt].[md].[B2B_cohort_funddt_staging]


## 10. Assign Cohorts from the Static `CohortBaseDt` Attributes

**Performance & invariance design:** rather than scoring 11.8M snapshot rows, pull each distinct loan's
`_Base` attributes **once**, run `assign_cohorts()`, build `Cohort_ID` / `cohort_unique_ID`, stage the
result, then fan it out to every snapshot row for that `LoanNum` in a single SQL `UPDATE`.

Cohort inputs (all as of `CohortBaseDt`):
- `LoanType_Base` (normalized to `CONV` / `FHA` / `VA` / `OTHERS`), `RemainingTerm_Base`
- `FICOOrig_Base`, `PrinBal_Base`, `IntRate_Base`, `Market_Rate_Base`
- `RateSpread = IntRate_Base − Market_Rate_Base`

`InvestorCatgy` and `LoanAge` are no longer cohort inputs (they remain in the table).
Loans missing `PrinBal_Base`, `IntRate_Base` or `Market_Rate_Base` cannot be scored and are left unassigned.

This guarantees every loan carries one static cohort across all `AsofDt` months.


In [11]:
def _to_py(val):
    if val is None:
        return None
    if isinstance(val, float) and (val != val):
        return None
    if hasattr(val, 'item'):
        return val.item()
    return val

# ── Mode-based setup ──────────────────────────────────────────────────────────
if RUN_MODE == 'COHORT_REFRESH':
    print("COHORT_REFRESH: Clearing all cohort columns in main table ...")
    _cur = cn.cursor()
    _cur.execute(f"""
        UPDATE {FULL_TABLE_NAME}
        SET Rule_Bucket      = NULL,
            Mean_CPR         = NULL,
            Std_CPR          = NULL,
            Cohort_ID        = NULL,
            cohort_unique_ID = NULL
    """)
    print(f"  ✅ {_cur.rowcount:,} rows cleared.")
    _cur.close()
    LOAN_PULL_FILTER = ""
else:
    # APPEND or FULL_REBUILD — process loans where cohort is not yet assigned
    LOAN_PULL_FILTER = "WHERE Cohort_ID IS NULL"

# ── 1. One row per loan: baseline attributes (as of CohortBaseDt) ─────────────
print(f"Pulling distinct loans from {FULL_TABLE_NAME} ...")
t0 = time.time()
q_distinct_loans = f"""
SELECT
    LoanNum,
    CohortBaseDt,
    LoanType      = LoanType_Base,
    RemainingTerm = RemainingTerm_Base,
    FICOOrig      = FICOOrig_Base,
    PrinBal       = PrinBal_Base,
    IntRate       = IntRate_Base,
    Market_Rate   = Market_Rate_Base
FROM (
    SELECT
        LoanNum,
        CohortBaseDt,
        LoanType_Base,
        RemainingTerm_Base,
        FICOOrig_Base,
        PrinBal_Base,
        IntRate_Base,
        Market_Rate_Base,
        ROW_NUMBER() OVER (PARTITION BY LoanNum ORDER BY AsofDt DESC) AS rn
    FROM {FULL_TABLE_NAME}
    {LOAN_PULL_FILTER}
) sub
WHERE rn = 1
"""
df_loans = pd.read_sql(q_distinct_loans, cn)
print(f"  ✅ {len(df_loans):,} distinct loans retrieved in {time.time()-t0:.1f}s  (shape: {df_loans.shape})")

if not df_loans.empty:
    df_loans['LoanNum'] = df_loans['LoanNum'].str.strip()

    # ── 2. Run cohort assignment ──────────────────────────────────────────────
    print("Assigning cohorts from CohortBaseDt attributes (LoanType, RemainingTerm, FICOOrig, PrinBal, RateSpread) ...")
    t0 = time.time()
    df_assigned = assign_cohorts(df_loans)
    assert len(df_assigned) == len(df_loans), "Row count changed during cohort assignment"
    print(f"  ✅ Cohort assignment completed in {time.time()-t0:.1f}s")

    # ── 3. Build Cohort_ID & cohort_unique_ID ─────────────────────────────────
    base_yyyymm = pd.to_datetime(df_assigned['CohortBaseDt']).dt.strftime('%Y%m')

    df_assigned['Cohort_ID'] = np.where(
        df_assigned['RuleSeq'].notna(),
        base_yyyymm.fillna('999912') + '_C' + df_assigned['RuleSeq'].fillna('').astype(str),
        None
    )
    df_assigned['cohort_unique_ID'] = np.where(
        df_assigned['RuleSeq'].notna(),
        'C' + df_assigned['RuleSeq'].fillna('').astype(str),
        None
    )

    n_mapped   = int(df_assigned['Rule_Bucket'].notna().sum())
    n_unmapped = int(df_assigned['Rule_Bucket'].isna().sum())
    n_noCPR    = int((df_assigned['Rule_Bucket'].notna() & df_assigned['Mean_CPR'].isna()).sum())
    print(f"  Mapped loans   : {n_mapped:,}  ({n_mapped/len(df_loans)*100:.1f}%)")
    print(f"  Unmapped loans : {n_unmapped:,}  ({n_unmapped/len(df_loans)*100:.1f}%)  "
          f"[missing PrinBal / IntRate / Market_Rate baseline]")
    print(f"  Mapped to extrapolated cohort (no Mean_CPR) : {n_noCPR:,}")

    # ── 4. Bulk insert into staging table ─────────────────────────────────────
    print(f"\nInserting {len(df_assigned):,} rows into staging table {STAGING_TABLE} ...")
    t0 = time.time()
    stage_cols = ['LoanNum', 'Rule_Bucket', 'Mean_CPR', 'Std_CPR', 'Cohort_ID', 'cohort_unique_ID']
    stage_rows = [
        tuple(_to_py(v) for v in row)
        for row in df_assigned[stage_cols].itertuples(index=False, name=None)
    ]

    cursor = cn.cursor()
    cursor.fast_executemany = True
    cursor.execute(f"TRUNCATE TABLE {STAGING_TABLE};")
    cursor.executemany(
        f"INSERT INTO {STAGING_TABLE} (LoanNum, Rule_Bucket, Mean_CPR, Std_CPR, Cohort_ID, cohort_unique_ID) VALUES (?, ?, ?, ?, ?, ?)",
        stage_rows
    )
    cursor.close()
    print(f"  ✅ Staging insert complete in {time.time()-t0:.1f}s")
else:
    print("  ℹ️  No unassigned loans to process.")


Pulling distinct loans from [Analytics_Rpt].[md].[B2B_revamp_cohorts] ...
  ✅ 567,351 distinct loans retrieved in 13.5s  (shape: (567351, 8))
Assigning cohorts from CohortBaseDt attributes (LoanType, RemainingTerm, FICOOrig, PrinBal, RateSpread) ...
  ✅ Cohort assignment completed in 3.7s
  Mapped loans   : 567,075  (100.0%)
  Unmapped loans : 276  (0.0%)  [missing PrinBal / IntRate / Market_Rate baseline]
  Mapped to extrapolated cohort (no Mean_CPR) : 0

Inserting 567,351 rows into staging table [Analytics_Rpt].[md].[B2B_cohort_funddt_staging] ...
  ✅ Staging insert complete in 619.9s


## 11. Update Main Table from Staging

1. Create a clustered index on `STAGING_TABLE(LoanNum)`.
2. Execute a single `UPDATE ... FROM ... JOIN` on `LoanNum` to write cohort outputs to all snapshot rows.
3. Drop the staging table.

In [12]:
cursor = cn.cursor()

print("Step 1: Creating clustered index on staging table (LoanNum)...")
cursor.execute(f"""
CREATE CLUSTERED INDEX IX_staging_funddt_LoanNum ON {STAGING_TABLE} (LoanNum);
""")
print("        ✅ Done")

print(f"Step 2: Updating {FULL_TABLE_NAME} from staging table...")
t0 = time.time()
cursor.execute(f"""
UPDATE  t
SET     t.Rule_Bucket      = s.Rule_Bucket,
        t.Mean_CPR         = s.Mean_CPR,
        t.Std_CPR          = s.Std_CPR,
        t.Cohort_ID        = s.Cohort_ID,
        t.cohort_unique_ID = s.cohort_unique_ID
FROM    {FULL_TABLE_NAME} t
JOIN    {STAGING_TABLE}   s
    ON  t.LoanNum = s.LoanNum;
""")
rows_updated = cursor.rowcount
elapsed_upd  = time.time() - t0
print(f"        ✅ {rows_updated:,} rows updated in {elapsed_upd:.1f}s")

print("Step 3: Dropping staging table...")
cursor.execute(f"DROP TABLE {STAGING_TABLE};")
print(f"        ✅ {STAGING_TABLE} dropped")

cursor.close()
print(f"\n✅ Main table {FULL_TABLE_NAME} is fully updated with FundDt-based cohorts.")

Step 1: Creating clustered index on staging table (LoanNum)...
        ✅ Done
Step 2: Updating [Analytics_Rpt].[md].[B2B_revamp_cohorts] from staging table...
        ✅ 12,258,573 rows updated in 116.7s
Step 3: Dropping staging table...
        ✅ [Analytics_Rpt].[md].[B2B_cohort_funddt_staging] dropped

✅ Main table [Analytics_Rpt].[md].[B2B_revamp_cohorts] is fully updated with FundDt-based cohorts.


## 12. Validation & Invariance Verification

1. **Cohort null audit** — assignment rate across all snapshot rows.
2. **Loan-level invariance (critical)** — `COUNT(DISTINCT cohort_unique_ID) = 1` for every loan.
3. **Assignment by channel & anchor type** — mapping rate, cohort count and baseline rate spread.
4. **Distribution by `InvestorCatgy`** and by baseline `LoanType`.
5. **Spot-check** — one `LockDt`-anchored (Correspondent) and one `FundDt`-anchored loan, showing
   display attributes moving with `AsofDt` while the `_Base` attributes and cohort stay fixed.


In [13]:
# ── Check 1: Null audit for cohort columns ───────────────────────────────────
q_cnull = f"""
SELECT
    COUNT(*) AS TotalRows,
    SUM(CASE WHEN Rule_Bucket      IS NULL THEN 1 ELSE 0 END) AS Rule_Bucket_Null,
    SUM(CASE WHEN Mean_CPR         IS NULL THEN 1 ELSE 0 END) AS Mean_CPR_Null,
    SUM(CASE WHEN Std_CPR          IS NULL THEN 1 ELSE 0 END) AS Std_CPR_Null,
    SUM(CASE WHEN Cohort_ID        IS NULL THEN 1 ELSE 0 END) AS Cohort_ID_Null,
    SUM(CASE WHEN cohort_unique_ID IS NULL THEN 1 ELSE 0 END) AS cohort_unique_ID_Null
FROM {FULL_TABLE_NAME}
"""
df_cn = pd.read_sql(q_cnull, cn)
total = df_cn.at[0, 'TotalRows']
print("CHECK 1 — Cohort column null audit:")
for c in ['Rule_Bucket', 'Mean_CPR', 'Std_CPR', 'Cohort_ID', 'cohort_unique_ID']:
    n   = df_cn.at[0, f'{c}_Null']
    pct = n / total * 100 if total > 0 else 0
    print(f"  {c:<18}: {n:>9,} NULLs ({pct:.2f}%)  "
          f"{'← missing baseline PrinBal / IntRate / Market_Rate (expected, small)' if c == 'Rule_Bucket' else ''}")

# ── Check 2: Loan-level Invariance Check (CRITICAL) ───────────────────────────
q_invariance = f"""
SELECT COUNT(*) AS Loans_With_Multiple_Cohorts
FROM (
    SELECT LoanNum
    FROM {FULL_TABLE_NAME}
    WHERE cohort_unique_ID IS NOT NULL
    GROUP BY LoanNum
    HAVING COUNT(DISTINCT cohort_unique_ID) > 1
) invalid_loans
"""
df_inv_check = pd.read_sql(q_invariance, cn)
multi_cohort_count = df_inv_check.at[0, 'Loans_With_Multiple_Cohorts']

print(f"\nCHECK 2 — Loan-level cohort invariance across AsofDt:")
print(f"  Loans with multiple cohorts: {multi_cohort_count}")
print("  ✅ PASS — every loan keeps one cohort_unique_ID across all snapshot months."
      if multi_cohort_count == 0 else
      "  ❌ FAIL — detected loans with varying cohorts across months!")

# ── Check 3: Cohort assignment rate by Channel & anchor-date type ────────────
q_dist = f"""
SELECT
    Channel,
    CohortBaseDtType,
    COUNT(DISTINCT LoanNum)                                                     AS Loans,
    COUNT(DISTINCT CASE WHEN Rule_Bucket IS NOT NULL THEN LoanNum END)          AS Mapped,
    COUNT(DISTINCT cohort_unique_ID)                                            AS UniqueCohorts,
    AVG(Mean_CPR)                                                               AS Avg_Mean_CPR,
    AVG(Market_Rate_Base)                                                       AS Avg_Market_Rate_Base,
    AVG(IntRate_Base - Market_Rate_Base)                                        AS Avg_RateSpread_Base
FROM {FULL_TABLE_NAME}
GROUP BY Channel, CohortBaseDtType
ORDER BY Loans DESC
"""
df_dist = pd.read_sql(q_dist, cn)
df_dist['Mapped_Pct']           = (df_dist['Mapped'] / df_dist['Loans'] * 100).map('{:.1f}%'.format)
df_dist['Avg_Mean_CPR']         = df_dist['Avg_Mean_CPR'].map(lambda x: f'{x:.2f}%' if pd.notna(x) else 'N/A')
df_dist['Avg_Market_Rate_Base'] = df_dist['Avg_Market_Rate_Base'].map(lambda x: f'{x:.2f}%' if pd.notna(x) else 'N/A')
df_dist['Avg_RateSpread_Base']  = df_dist['Avg_RateSpread_Base'].map(lambda x: f'{x:+.2f}%' if pd.notna(x) else 'N/A')
df_dist['Loans']                = df_dist['Loans'].map('{:,}'.format)
df_dist['Mapped']               = df_dist['Mapped'].map('{:,}'.format)
print("\nCHECK 3 — Cohort assignment by Channel & anchor-date type:")
print(df_dist.to_string(index=False))

# ── Check 4: Distribution by InvestorCatgy and by baseline LoanType ──────────
q_inv = f"""
SELECT
    ISNULL(InvestorCatgy, 'UNKNOWN') AS InvestorCatgy,
    COUNT(*)                         AS NumRows,
    COUNT(DISTINCT LoanNum)          AS UniqueLoans,
    COUNT(DISTINCT cohort_unique_ID) AS UniqueCohorts,
    AVG(Mean_CPR)                    AS Avg_Mean_CPR
FROM {FULL_TABLE_NAME}
GROUP BY InvestorCatgy
ORDER BY NumRows DESC
"""
df_inv = pd.read_sql(q_inv, cn)
df_inv['Avg_Mean_CPR'] = df_inv['Avg_Mean_CPR'].map(lambda x: f'{x:.2f}%' if pd.notna(x) else 'N/A')
df_inv['NumRows']      = df_inv['NumRows'].map('{:,}'.format)
df_inv['UniqueLoans']  = df_inv['UniqueLoans'].map('{:,}'.format)
print("\nCHECK 4a — Cohort distribution by InvestorCatgy (as of AsofDt):")
print(df_inv.to_string(index=False))

q_lt = f"""
SELECT
    ISNULL(LoanType_Base, 'UNKNOWN') AS LoanType_Base,
    COUNT(DISTINCT LoanNum)          AS UniqueLoans,
    COUNT(DISTINCT cohort_unique_ID) AS UniqueCohorts,
    AVG(Mean_CPR)                    AS Avg_Mean_CPR
FROM {FULL_TABLE_NAME}
GROUP BY LoanType_Base
ORDER BY UniqueLoans DESC
"""
df_lt = pd.read_sql(q_lt, cn)
df_lt['Avg_Mean_CPR'] = df_lt['Avg_Mean_CPR'].map(lambda x: f'{x:.2f}%' if pd.notna(x) else 'N/A')
df_lt['UniqueLoans']  = df_lt['UniqueLoans'].map('{:,}'.format)
print("\nCHECK 4b — Cohort distribution by raw LoanType_Base (normalized to CONV/FHA/VA/OTHERS at scoring):")
print(df_lt.to_string(index=False))

# ── Check 5: Spot-check one Correspondent (LockDt) and one Co-Issue (FundDt) ──
spot_cols = ['LoanNum', 'AsofDt', 'Channel', 'CohortBaseDtType', 'LockDt', 'FundDt',
             'CohortBaseDt', 'Base_AsofDt', 'PrinBal', 'LoanAge', 'RemainingTerm', 'LoanType',
             'IntRate', 'Market_Rate',
             'PrinBal_Base', 'LoanAge_Base', 'RemainingTerm_Base', 'LoanType_Base', 'IntRate_Base',
             'Market_Rate_Base', 'Market_Rate_BaseDt', 'cohort_unique_ID', 'Mean_CPR']

for _type in ['LockDt', 'FundDt']:
    q_pick = f"""
    SELECT TOP 1 LoanNum
    FROM {FULL_TABLE_NAME}
    WHERE cohort_unique_ID IS NOT NULL AND CohortBaseDtType = '{_type}'
    GROUP BY LoanNum
    HAVING COUNT(DISTINCT AsofDt) >= 12
    """
    _df_pick = pd.read_sql(q_pick, cn)
    if _df_pick.empty:
        print(f"\nCHECK 5 — No loan found with CohortBaseDtType = '{_type}'")
        continue
    _loan = _df_pick.at[0, 'LoanNum']

    q_spot = f"""
    SELECT TOP 4 {', '.join(spot_cols)}
    FROM {FULL_TABLE_NAME}
    WHERE LoanNum = '{_loan}'
    ORDER BY AsofDt DESC
    """
    _d = pd.read_sql(q_spot, cn)
    for _c in ['AsofDt', 'LockDt', 'FundDt', 'CohortBaseDt', 'Base_AsofDt', 'Market_Rate_BaseDt']:
        _d[_c] = pd.to_datetime(_d[_c]).dt.strftime('%Y-%m-%d')
    for _c in ['PrinBal', 'PrinBal_Base']:
        _d[_c] = _d[_c].map(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
    _d['Mean_CPR'] = _d['Mean_CPR'].map(lambda x: f'{x:.2f}%' if pd.notna(x) else 'N/A')

    print(f"\nCHECK 5 — Spot-check, anchor = {_type} (LoanNum = {_loan}):")
    print(_d[['AsofDt', 'Channel', 'CohortBaseDtType', 'LockDt', 'FundDt', 'CohortBaseDt',
              'Base_AsofDt', 'PrinBal', 'LoanAge', 'RemainingTerm', 'LoanType', 'IntRate', 'Market_Rate']].to_string(index=False))
    print(_d[['AsofDt', 'PrinBal_Base', 'LoanAge_Base', 'RemainingTerm_Base', 'LoanType_Base', 'IntRate_Base',
              'Market_Rate_Base', 'Market_Rate_BaseDt', 'cohort_unique_ID', 'Mean_CPR']].to_string(index=False))


CHECK 1 — Cohort column null audit:
  Rule_Bucket       :     6,456 NULLs (0.05%)  ← missing baseline PrinBal / IntRate / Market_Rate (expected, small)
  Mean_CPR          :     6,456 NULLs (0.05%)  
  Std_CPR           :     6,456 NULLs (0.05%)  
  Cohort_ID         :     6,456 NULLs (0.05%)  
  cohort_unique_ID  :     6,456 NULLs (0.05%)  

CHECK 2 — Loan-level cohort invariance across AsofDt:
  Loans with multiple cohorts: 0
  ✅ PASS — every loan keeps one cohort_unique_ID across all snapshot months.

CHECK 3 — Cohort assignment by Channel & anchor-date type:
        Channel CohortBaseDtType   Loans  Mapped  UniqueCohorts Avg_Mean_CPR Avg_Market_Rate_Base Avg_RateSpread_Base Mapped_Pct
       Co-Issue           FundDt 279,215 279,213            104        9.33%                4.51%              -0.22%     100.0%
  Correspondent           LockDt 256,228 255,964            106        9.55%                5.27%              -0.30%      99.9%
Consumer Direct           FundDt  31,906  31

## 13. Key Takeaways & Close Connection

In [14]:
print("=" * 78)
print(f"  TABLE BUILD SUMMARY: {FULL_TABLE_NAME}")
print("=" * 78)
print(f"  Rows              : {actual_rows:,}")
print(f"  Distinct loans    : {distinct_loans:,}")
print(f"  Clustered index   : (LoanNum, AsofDt)")
print()
print("  Cohort anchor date (CohortBaseDt):")
print(f"    Correspondent   -> LockDt from vwOrigLoan (fallback FundDt)")
print(f"    Co-Issue        -> FundDt")
print(f"    Consumer Direct -> FundDt")
print(f"    Baseline snapshot read at EOMONTH(CohortBaseDt), cascading up to "
      f"+{MAX_BASE_FALLBACK_MONTHS} months -> Base_AsofDt")
print()
print(f"  Market rate       : blend of {', '.join(RATE_SERIES)}")
print( "                      latest observation per series on/before the anchor date")
print()
print("  Column groups:")
print("    Display  (AsofDt)       : FICOOrig, PrinBal, IntRate, Market_Rate, InvestorCatgy, LoanAge,")
print("                              RemainingTerm, LoanType")
print("    Transparency            : Channel, FundDt, LockDt, CohortBaseDt, CohortBaseDtType, Base_AsofDt")
print("    Baseline (CohortBaseDt) : FICOOrig_Base, PrinBal_Base, IntRate_Base, Market_Rate_Base,")
print("                              Market_Rate_BaseDt, InvestorCatgy_Base, LoanAge_Base,")
print("                              RemainingTerm_Base, LoanType_Base")
print("    Cohort                  : Rule_Bucket, Mean_CPR, Std_CPR, Cohort_ID, cohort_unique_ID")
print()
print("  Cohort inputs (schema v4): LoanType, RemainingTerm, FICOOrig, PrinBal, RateSpread (all _Base).")
print("  Cohorts are assigned from the baseline attributes only, so each loan keeps the")
print("  same cohort across every historical and future AsofDt snapshot.")
print("=" * 78)

try:
    cn.close()
    print("\n✅ Analytics_Rpt connection closed")
except Exception:
    pass


  TABLE BUILD SUMMARY: [Analytics_Rpt].[md].[B2B_revamp_cohorts]
  Rows              : 12,258,573
  Distinct loans    : 567,351
  Clustered index   : (LoanNum, AsofDt)

  Cohort anchor date (CohortBaseDt):
    Correspondent   -> LockDt from vwOrigLoan (fallback FundDt)
    Co-Issue        -> FundDt
    Consumer Direct -> FundDt
    Baseline snapshot read at EOMONTH(CohortBaseDt), cascading up to +3 months -> Base_AsofDt

  Market rate       : blend of MORTGAGE30US, OBMMIC30YF, MND_30Y
                      latest observation per series on/before the anchor date

  Column groups:
    Display  (AsofDt)       : FICOOrig, PrinBal, IntRate, Market_Rate, InvestorCatgy, LoanAge,
                              RemainingTerm, LoanType
    Transparency            : Channel, FundDt, LockDt, CohortBaseDt, CohortBaseDtType, Base_AsofDt
    Baseline (CohortBaseDt) : FICOOrig_Base, PrinBal_Base, IntRate_Base, Market_Rate_Base,
                              Market_Rate_BaseDt, InvestorCatgy_Base, LoanA